# 静的メンバー

## 概要

静的メンバー（static members）は、クラスのインスタンスではなく、クラス自体に属するメンバーです。静的メンバーはアプリケーション全体で共有され、インスタンスを作成せずにアクセスできます。

## 静的メンバーの種類

### 1. 静的フィールド

クラスの全インスタンスで共有される変数です。

In [1]:
public class Counter
{
    // 全インスタンスで共有されるカウント
    private static int _totalCount = 0;
    
    // インスタンスごとの個別のID
    private readonly int _id;
    
    public Counter()
    {
        _totalCount++;
        _id = _totalCount;
    }
    
    public static int TotalCount => _totalCount;
    public int Id => _id;
}

### 2.　静的プロパティ

クラスレベルでの状態管理や設定値の保持に使用します。

In [2]:
public class ApplicationSettings
{
    private static string _environment = "Development";
    
    public static string Environment
    {
        get => _environment;
        set => _environment = value?.ToUpper() ?? "Development";
    }
    
    public static bool IsProduction => Environment == "PRODUCTION";
    public static bool IsDevelopment => Environment == "DEVELOPMENT";
}

### 3. 静的メソッド

インスタンス状態に依存しない処理を実装します。

In [3]:
public class MathHelper
{
    public static double CalculateCircleArea(double radius)
    {
        if (radius < 0)
            throw new ArgumentException("半径は負の値にはできません。", nameof(radius));
            
        return Math.PI * radius * radius;
    }
    
    public static double CalculateCircleCircumference(double radius)
    {
        if (radius < 0)
            throw new ArgumentException("半径は負の値にはできません。", nameof(radius));
            
        return 2 * Math.PI * radius;
    }
}

### 4. 静的コンストラクタ

クラスの初期化時に一度だけ実行される特別なコンストラクタです。

In [4]:
public class DatabaseConnection
{
    private static readonly string _connectionString;
    
    // 静的コンストラクタ
    static DatabaseConnection()
    {
        // 設定ファイルから接続文字列を読み込むなどの初期化処理
        _connectionString = LoadConnectionString();
    }
    
    private static string LoadConnectionString()
    {
        // 設定ファイルから読み込む処理
        return "Server=...;Database=...;";
    }
}

## 静的クラス

すべてのメンバーが静的で、インスタンス化できないクラスを定義できます。

※以下の例では、静的クラス内に拡張メソッドを定義しています。拡張メソッドは.NET Interactiveでは正しく実行できないのでエラーになりますが、Visual Studio上では正しく実行できます。

In [5]:
public static class StringExtensions
{
    public static bool IsNullOrWhiteSpace(this string value)
    {
        return string.IsNullOrWhiteSpace(value);
    }
    
    public static string ToCamelCase(this string value)
    {
        if (string.IsNullOrEmpty(value)) return value;
        return char.ToLowerInvariant(value[0]) + value.Substring(1);
    }
}

Error: (3,24): error CS1109: 拡張メソッドは、トップ レベルの静的クラスで定義される必要があります。StringExtensions は入れ子にされたクラスです
(8,26): error CS1109: 拡張メソッドは、トップ レベルの静的クラスで定義される必要があります。StringExtensions は入れ子にされたクラスです

## 一般的な使用パターン

### 1. ユーティリティ関数

In [6]:
public static class DateTimeHelper
{
    public static bool IsBusinessDay(DateTime date)
    {
        return date.DayOfWeek != DayOfWeek.Saturday 
            && date.DayOfWeek != DayOfWeek.Sunday;
    }
    
    public static DateTime NextBusinessDay(DateTime date)
    {
        var next = date.AddDays(1);
        while (!IsBusinessDay(next))
        {
            next = next.AddDays(1);
        }
        return next;
    }
}

### 2. ファクトリーメソッド

In [7]:
public enum UserRole
{
    Admin,
    Regular
}

public class User
{
    public string Name { get; }
    public string Email { get; }
    public UserRole Role { get; }
    
    private User(string name, string email, UserRole role)
    {
        Name = name;
        Email = email;
        Role = role;
    }
    
    public static User CreateAdmin(string name, string email)
    {
        return new User(name, email, UserRole.Admin);
    }
    
    public static User CreateRegularUser(string name, string email)
    {
        return new User(name, email, UserRole.Regular);
    }
}

### 3. キャッシュとシングルトン

In [8]:
public class Configuration
{
    private static Configuration _instance;
    private static readonly object _lock = new object();
    
    private Dictionary<string, string> _settings;
    
    private Configuration()
    {
        _settings = LoadSettings();
    }
    
    public static Configuration Instance
    {
        get
        {
            if (_instance == null)
            {
                lock (_lock)
                {
                    _instance ??= new Configuration();
                }
            }
            return _instance;
        }
    }
    
    private Dictionary<string, string> LoadSettings()
    {
        // 設定の読み込み処理
        return new Dictionary<string, string>();
    }
}

## 注意点

1. 静的メンバーは共有されるため、マルチスレッド環境での同期に注意
1. 静的メソッドはオーバーライドできない
1. 静的メンバーはメモリに常駐するため、必要な場合のみ使用
1. 単体テストが難しくなる可能性がある

## 演習問題

### 問題１：ロギングユーティリティ

以下の要件を満たすロギングユーティリティを実装してください：

In [ ]:
public static class Logger
{
    // TODO: ログレベル（Debug, Info, Warning, Error）を定義
    // TODO: ログメッセージを保存するための静的コレクション
    // TODO: ログを追加するメソッド
    // TODO: 特定のログレベルのメッセージのみを取得するメソッド
    // TODO: ログをクリアするメソッド
}

### 問題２：通貨変換器

以下の要件を満たす通貨変換器を実装してください：

In [ ]:
public static class CurrencyConverter
{
    // TODO: 通貨レートを保持する静的ディクショナリ
    // TODO: 通貨レートを追加・更新するメソッド
    // TODO: ある通貨から別の通貨に変換するメソッド
    // TODO: サポートされている通貨のリストを取得するメソッド
}

### 問題３：キャッシュマネージャー

以下の要件を満たすキャッシュマネージャーを実装してください：

In [ ]:
public static class CacheManager
{
    // TODO: キャッシュを保持する静的ディクショナリ
    // TODO: キャッシュにデータを追加するメソッド（有効期限付き）
    // TODO: キャッシュからデータを取得するメソッド
    // TODO: 期限切れのキャッシュを削除するメソッド
    // TODO: キャッシュの統計情報を取得するメソッド
}

## 解答例

各演習問題の解答例は別途提供しますが、まずは自力で実装にチャレンジしてください。実装時は以下の点を考慮してください：

1. スレッドセーフティ
1. パフォーマンス
1. メモリ使用量
1. エラーハンドリング
1. テスタビリティ

## まとめ

静的メンバーは以下のような場合に適しています：

* ユーティリティ関数の提供
* アプリケーション全体での状態管理
* 共通の処理やリソースの共有
* ファクトリーメソッドの実装
* 拡張メソッドの定義

ただし、過度な使用は避け、インスタンスメンバーとの適切な使い分けを心がけましょう。